In [57]:
import io
import json
import zipfile
from pathlib import Path
from typing import Literal
from urllib.parse import quote

import httpx
from tqdm.auto import tqdm
from minsearch import Index
from pydantic import BaseModel
from anthropic import Anthropic
from gitsource import chunk_documents

In [4]:
client = Anthropic()

INSTRUCTIONS = """
You're an IaC (Infrastructure as Code) assistant, your task is to answer the QUESTION from the
documentation that developers and/or engineers will be asking using the provided CONTEXT.
"""

PROMPT_TEMPLATE = """
<QUESTION>
{question}
</QUESTION>

<CONTEXT>
{context}
</CONTEXT>
""".strip()

In [5]:
class RAGResponse(BaseModel):
    answer: str
    found_answer: bool
    confidence: float
    confidence_explanation: str
    answer_type: Literal["how-to", "explanation", "troubleshooting", "comparison", "reference"]
    followup_questions: list[str]

In [61]:
class RAG:
    def __init__(
        self,
        index: Index,
        llm_client: Anthropic,
        output_type: type[BaseModel],
        instructions: str = INSTRUCTIONS,
    ):
        self.index = index
        self.llm_client = llm_client
        self.output_type = output_type
        self.instructions = instructions

    def search(self, query: str, top_k: int = 5):
        return self.index.search(query, num_results=top_k)

    def build_prompt(self, question: str, search_results: list):
        context = json.dumps(search_results, indent=2)
        return PROMPT_TEMPLATE.format(
            question=question,
            context=context
        )

    def llm(
        self,
        user_prompt: str,
        max_tokens: int,
        model: str = "claude-opus-5",
    ):
        with self.llm_client.messages.stream(
            model=model,
            max_tokens=max_tokens,
            messages=[
                {"role": "assistant", "content": self.instructions or "You are a helpful course assistant."},
                {"role": "user", "content": user_prompt}
            ],
            thinking={"type": "adaptive", "display": "summarized"},
            output_format=self.output_type,
        ) as stream:
            response = stream.get_final_message()
            return response            

    def invoke(self, query: str, max_tokens: int = 2056):
        search_results = self.search(query=query)
        user_prompt = self.build_prompt(question=query, search_results=search_results)
        response = self.llm(user_prompt=user_prompt, max_tokens=max_tokens)
        
        output = None
        for block in response.content:
            if block.type == "text":
                output = block.parsed_output
        return output

In [47]:
def get_or_download_documentation(
    repo_owner: str,
    repo_name: str,
    branch_name: str,
    docs_path: str = "docs",
    save_dir: str | None = None,
) -> bytes:
    """
    Return the Markdown documentation as a zip archive, reusing a local copy if one already exists, and downloading it otherwise.

    Args:
        repo_owner (str): The owner of the GitHub repository.
        repo_name (str): The name of the GitHub repository.
        branch_name (str): The branch to download from.
        docs_path (str): Path of the documentation folder inside the repository.
        save_dir (str | None): Local directory to reuse/save the files in. If
            None, the files are only returned as an in-memory zip.

    Returns:
        bytes: A zip archive containing only the Markdown files under `docs_path`.
    """

    def _get(doc_dir: str | None = None) -> bytes | None:
        """Build a zip from an already-downloaded local directory.

        Args:
            doc_dir (str | None): Directory that may already contain the files.

        Returns:
            bytes | None: A zip archive of the local files, or None if the
            directory is missing or empty.
        """
        if doc_dir is None:
            return None

        out_dir = Path(doc_dir)
        existing_files = sorted(out_dir.rglob("*")) if out_dir.exists() else []
        existing_files = [f for f in existing_files if f.is_file()]
        if not existing_files:
            return None

        print(f"Using already-downloaded documentation in '{doc_dir}' ({len(existing_files)} files)")

        buffer = io.BytesIO()
        with zipfile.ZipFile(buffer, "w") as archive:
            for file in existing_files:
                archive.write(file, file.relative_to(out_dir).as_posix())
        return buffer.getvalue()

    def _download() -> bytes:
        """Download only the Markdown documentation from GitHub.

        Returns:
            bytes: A zip archive containing only the Markdown files under
            `docs_path`.
        """
        api = "https://api.github.com"
        raw = "https://raw.githubusercontent.com"

        with httpx.Client(follow_redirects=True) as client:
            # Resolve the branch to its root tree SHA
            branch = client.get(f"{api}/repos/{repo_owner}/{repo_name}/branches/{branch_name}")
            branch.raise_for_status()
            root_tree_sha = branch.json()["commit"]["commit"]["tree"]["sha"]
            print(f"Retrieved root tree SHA: {root_tree_sha}")

            # Locate the docs subtree inside the root tree
            root_tree = client.get(f"{api}/repos/{repo_owner}/{repo_name}/git/trees/{root_tree_sha}")
            root_tree.raise_for_status()
            docs_entry = next(
                (item for item in root_tree.json()["tree"] if item["path"] == docs_path),
                None,
            )
            if docs_entry is None:
                raise ValueError(f"Path '{docs_path}' not found in {repo_owner}/{repo_name}@{branch_name}")
            print(f"Retrieved docs subtree '{docs_path}' (SHA: {docs_entry['sha']})")

            # Recursively list every file under docs/
            docs_tree = client.get(
                f"{api}/repos/{repo_owner}/{repo_name}/git/trees/{docs_entry['sha']}?recursive=1"
            )
            docs_tree.raise_for_status()
            print(f"Retrieved recursive tree for '{docs_path}'")

            # Collect the Markdown files to download
            markdown_files = [
                item["path"]
                for item in docs_tree.json().get("tree", [])
                if item.get("type") == "blob" and item["path"].endswith((".md", ".mdx"))
            ]
            print(f"Found {len(markdown_files)} Markdown files to download")

            # Download each Markdown file and package it into an in-memory zip
            buffer = io.BytesIO()
            with zipfile.ZipFile(buffer, "w") as archive:
                for path in tqdm(markdown_files, desc="Downloading docs", unit="file"):
                    file_url = f"{raw}/{repo_owner}/{repo_name}/{branch_name}/{quote(f'{docs_path}/{path}')}"
                    response = client.get(file_url)
                    response.raise_for_status()
                    archive.writestr(path, response.content)

        return buffer.getvalue()

    # Reuse the local copy if it already exists
    data = _get(save_dir)
    if data is not None:
        return data

    # Otherwise download and (optionally) save to disk
    data = _download()

    if save_dir is not None:
        out_dir = Path(save_dir)
        out_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(io.BytesIO(data)) as zf:
            num_files = len(zf.namelist())
            zf.extractall(out_dir)
        print(f"Extracted {num_files} files to '{save_dir}'")

    return data

In [65]:
def retrieve_documentation(
    repo_owner: str,
    repo_name: str,
    branch_name: str,
    docs_path: str = "docs",
    save_dir: str | None = None,
):
    response_contents = get_or_download_documentation(
        repo_owner=repo_owner,
        repo_name=repo_name,
        branch_name=branch_name,
        docs_path=docs_path,
        save_dir=save_dir,
    )
    documents = []
    with zipfile.ZipFile(io.BytesIO(response_contents)) as zip_ref:
        print("Zip Ref: ", zip_ref.namelist())
        for file_path in zip_ref.namelist():
            with zip_ref.open(file_path) as file:
                content = file.read().decode('utf-8')
                doc = {
                    'content': content,
                    'filename': file_path
                }
                documents.append(doc)
    return documents

In [66]:
documents = retrieve_documentation(
    repo_owner="pulumi",
    repo_name="docs",
    branch_name="master",
    docs_path="content",
    save_dir="docs",
)
documents[:5]

Using already-downloaded documentation in 'docs' (2186 files)
Zip Ref:  ['_index.md', 'about/_index.md', 'about/leadership.md', 'about/newsroom.md', 'agent-start/_index.md', 'authors/_index.md', 'automation/_index.md', 'awards/_index.md', 'awsx/_index.md', 'b/_index.md', 'b/iac-platform-engineering-idp.md', 'blog/10-more-things-you-can-do-with-neo/index.md', 'blog/10-things-you-can-do-with-neo/index.md', 'blog/10-tips-to-improve-your-coding-agent-game/index.md', 'blog/2018-year-at-a-glance/index.md', 'blog/2019-year-at-a-glance/index.md', 'blog/2021-dec-hackathon/index.md', 'blog/2021-end-of-year-review/index.md', 'blog/2022-03-10-hierarchical-config/index.md', 'blog/2022-03-30-introducing-pulumiverse/index.md', 'blog/2022-03-31-go-generics-with-pulumi/index.md', 'blog/2025-product-launches/index.md', 'blog/7-ways-to-deal-with-application-secrets-in-azure/index.md', 'blog/_index.md', 'blog/accelerate-revenue-growth-time-market-with-platform-engineering/index.md', 'blog/access-token-exp

[{'content': '---\ntitle: The infrastructure as code platform for the AI era – Pulumi\nmeta_desc: Pulumi is an open-source infrastructure as code platform that helps humans and agents build and manage cloud infrastructure with real programming languages.\nmeta_image: /images/home/home-meta.png\n\ninclude_organization_schema: true\n\nsections:\n  - type: hero\n    layout: split\n    cta_secondary_text: Download open source\n    cta_secondary_link: /docs/install/\n    badge_highlight_text: "New:"\n    badge_text: "Native support for Terraform and HCL"\n    badge_link: /releases/terraform-state-backend-modules-hcl/\n    title: "*Unleash agents* <br>on your infrastructure."\n    description: |\n      Ship infrastructure faster with tools that let agents do what they do best — without letting them go off the rails.\n    anchor: hero\n    hero_animation: agent-loop\n\n  - type: logo_grid\n    title: Trusted by over 4,000 innovative companies\n    logos:\n      - snowflake\n      - nvidia\n  

In [58]:
document_chunks = chunk_documents(documents, size=3000, step=1500)

In [59]:
index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
index.fit(document_chunks)

In [67]:
rag = RAG(
    index=index,
    llm_client=client,
    output_type=RAGResponse,
)

In [68]:
response = rag.invoke(
    query="How to look up an existing AMI (e.g. the latest Amazon Linux or Ubuntu image) from within an IaC program?",
    max_tokens=8192,
)
print(f"Answer: {response.answer}")  # type: ignore
print(f"Found answer: {response.found_answer}")  # type: ignore
print(f"Confidence: {response.confidence}")  # type: ignore
print(f"Answer type: {response.answer_type}")  # type: ignore
print(f"Follow-up questions: {response.followup_questions}")  # type: ignore

Answer: ## Looking up an existing AMI with Pulumi

AMIs aren't something you create in your program — you query them using the AWS provider's `getAmi` **invoke (function/data source)**, filter on the image name, pin the `owners`, and set `mostRecent: true` so you always get the newest matching image. Then you pass the returned `id` into your `ec2.Instance`.

### TypeScript

```typescript
import * as aws from "@pulumi/aws";

// Look up the latest Amazon Linux 2 AMI.
const ami = aws.ec2.getAmiOutput({
    owners: ["amazon"],
    mostRecent: true,
    filters: [{ name: "name", values: ["amzn2-ami-hvm-*-x86_64-gp2"] }],
});

const group = new aws.ec2.SecurityGroup("web-secgrp", {
    ingress: [{ protocol: "tcp", fromPort: 80, toPort: 80, cidrBlocks: ["0.0.0.0/0"] }],
});

export const server = new aws.ec2.Instance("web-server-www", {
    instanceType: "t2.micro",
    securityGroups: [group.name],
    ami: ami.id,
    tags: { Name: "webserver" },
});
```

There are two flavors in TypeScript

In [69]:
response = rag.invoke(
    query=response.followup_questions[0],
    max_tokens=8192,
)
print(f"Answer: {response.answer}")  # type: ignore
print(f"Found answer: {response.found_answer}")  # type: ignore
print(f"Confidence: {response.confidence}")  # type: ignore
print(f"Answer type: {response.answer_type}")  # type: ignore
print(f"Follow-up questions: {response.followup_questions}")  # type: ignore

Answer: ## Short answer

`getAmi` is the **direct form** of the provider function: it takes plain, already-known values and returns a language-native asynchronous/synchronous result (`Promise<GetAmiResult>` in TypeScript, `Task<...>` in .NET, `CompletableFuture<...>` in Java, plain synchronous returns in Python and Go).

`getAmiOutput` is the **output form**: it accepts Pulumi `Input`s and returns a `pulumi.Output<GetAmiResult>`, so it fits into the normal Pulumi programming model.

> Note on naming: not every language uses the `...Output` suffix. In Java, for example, `Ec2Functions.getAmi()` *is* the output form (it returns `Output<GetAmiResult>`), and in YAML there's no distinction at all — `fn::invoke` resolves all variable references before invoking the function.

## Key differences

| | Direct form (`getAmi`) | Output form (`getAmiOutput`) |
|---|---|---|
| Arguments | Plain values only (no Inputs/Outputs) | Accepts Pulumi `Input`s (including Outputs and secret config values) |
| 